# Histogram Target Sanity Check

Interactive exploration of the GBIF histogram targets Zarr store.
Verifies spatial coverage, per-trait validity, and distribution shapes.

In [ ]:
import re
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pyproj
import zarr

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Load data

In [ ]:
PROJECT_ROOT = Path("..")
ZARR_PATH = PROJECT_ROOT / "data/interim/histogram_targets/try6_hist_pow-xf_22km/gbif/histograms.zarr"
PARAMS_PATH = PROJECT_ROOT / "pipeline/histogram_data/try6_hist_pow-xf_22km/params.yaml"

root = zarr.open_group(ZARR_PATH, mode="r")
histograms = np.asarray(root["histograms"])  # (N, 31, 20)
masks = np.asarray(root["masks"])              # (N, 31)
coords = np.asarray(root["coords"])            # (N, 2) — EPSG:6933
bin_edges = np.asarray(root["bin_edges"])       # (31, 21)
attrs = dict(root.attrs)

trait_names = attrs["trait_names"]
n_cells, n_traits, n_bins = histograms.shape

print(f"Cells: {n_cells}, Traits: {n_traits}, Bins: {n_bins}")
print(f"Coords range: x=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}], y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}]")

In [ ]:
# Parse trait descriptions from params.yaml
trait_desc = {}
with open(PARAMS_PATH) as f:
    content = f.read()
for match in re.finditer(r"^\s*-\s*(X\d+)\s*#\s*(.+)$", content, re.MULTILINE):
    trait_desc[match.group(1)] = match.group(2).strip()

# Reproject coords to lon/lat for plotting
transformer = pyproj.Transformer.from_crs("EPSG:6933", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(coords[:, 0], coords[:, 1])

print(f"Lon range: [{lon.min():.1f}, {lon.max():.1f}]")
print(f"Lat range: [{lat.min():.1f}, {lat.max():.1f}]")

## 2. Spatial coverage map

Each dot is a 22km grid cell. Color = number of valid traits (out of 31).
Should show recognizable landmasses, with denser coverage where GBIF has more data.

In [ ]:
n_valid = masks.sum(axis=1)

fig = plt.figure(figsize=(18, 9))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.EqualEarth())
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", edgecolor="none")
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color="gray")

sc = ax.scatter(
    lon, lat, c=n_valid, s=1.5, cmap="viridis",
    transform=ccrs.PlateCarree(), rasterized=True,
)
cb = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.02)
cb.set_label("Valid traits per cell")
ax.set_title(f"GBIF histogram coverage ({n_cells:,} cells)")
plt.show()

## 3. Per-trait validity maps

Green = valid, red = invalid. Most traits should have high coverage,
but some (e.g., root traits, wood anatomy) may be sparser.

In [ ]:
ncols = 4
nrows = (n_traits + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows, ncols, figsize=(5 * ncols, 3 * nrows),
    subplot_kw={"projection": ccrs.EqualEarth()},
)
axes = np.asarray(axes).flatten()

for j, trait in enumerate(trait_names):
    ax = axes[j]
    ax.set_global()
    ax.add_feature(cfeature.COASTLINE, linewidth=0.2, color="gray")
    valid = masks[:, j]
    ax.scatter(lon[valid], lat[valid], c="#2ecc71", s=0.3, transform=ccrs.PlateCarree(), rasterized=True)
    ax.scatter(lon[~valid], lat[~valid], c="#e74c3c", s=0.1, alpha=0.3, transform=ccrs.PlateCarree(), rasterized=True)
    ax.set_title(f"{trait} ({int(valid.sum())})", fontsize=8)

for j in range(n_traits, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Per-trait validity", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 4. Histogram mean maps (key traits)

Weighted mean of each histogram: `sum(bin_centers * prob)`.
Should show plausible latitudinal / biome gradients.

In [ ]:
KEY_TRAITS = ["X4", "X14", "X3106", "X3117", "X26", "X6"]
key_idx = [(j, t) for j, t in enumerate(trait_names) if t in KEY_TRAITS]

ncols = 3
nrows = (len(key_idx) + ncols - 1) // ncols

fig, axes = plt.subplots(
    nrows, ncols, figsize=(6 * ncols, 4 * nrows),
    subplot_kw={"projection": ccrs.EqualEarth()},
)
axes = np.asarray(axes).flatten()

for i, (j, trait) in enumerate(key_idx):
    ax = axes[i]
    ax.set_global()
    ax.add_feature(cfeature.COASTLINE, linewidth=0.2, color="gray")

    valid = masks[:, j]
    edges = bin_edges[j]
    centers = 0.5 * (edges[:-1] + edges[1:])
    wmean = (histograms[valid, j, :] * centers[np.newaxis, :]).sum(axis=1)

    sc = ax.scatter(
        lon[valid], lat[valid], c=wmean, s=0.8, cmap="plasma",
        transform=ccrs.PlateCarree(), rasterized=True,
    )
    fig.colorbar(sc, ax=ax, shrink=0.5, pad=0.02)
    desc = trait_desc.get(trait, "")[:35]
    ax.set_title(f"{trait} — {desc}", fontsize=9)

for i in range(len(key_idx), len(axes)):
    axes[i].set_visible(False)

fig.suptitle("Histogram weighted mean by trait", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## 5. Entropy map

Shannon entropy averaged across valid traits. High entropy = near-uniform,
low entropy = peaked distributions. Helps spot suspicious cells.

In [ ]:
def shannon_entropy(h):
    """Shannon entropy in bits, ignoring zeros."""
    return -np.where(h > 0, h * np.log2(h), 0.0).sum(axis=-1)

# Entropy per (cell, trait)
ent = np.zeros((n_cells, n_traits), dtype=np.float32)
for j in range(n_traits):
    ent[:, j] = shannon_entropy(histograms[:, j, :])

# Mean across valid traits
masked_ent = np.where(masks, ent, np.nan)
mean_ent = np.nanmean(masked_ent, axis=1)

fig = plt.figure(figsize=(18, 9))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.EqualEarth())
ax.set_global()
ax.add_feature(cfeature.LAND, facecolor="#f0f0f0", edgecolor="none")
ax.add_feature(cfeature.COASTLINE, linewidth=0.3, color="gray")

sc = ax.scatter(
    lon, lat, c=mean_ent, s=1.5, cmap="coolwarm",
    transform=ccrs.PlateCarree(), rasterized=True,
)
cb = fig.colorbar(sc, ax=ax, shrink=0.6, pad=0.02)
cb.set_label("Mean Shannon entropy (bits)")
ax.set_title("Mean histogram entropy across valid traits")
plt.show()

## 6. Sample cell distributions

Pick one cell from each latitude band and plot histograms for key traits.
Green = valid trait, red = invalid (uniform prior).

In [ ]:
# Latitude bands (EPSG:6933 y-coordinates)
BANDS = {
    "Boreal": (5_500_000, 7_500_000),
    "N. Temperate": (2_500_000, 5_500_000),
    "N. Subtropical": (500_000, 2_500_000),
    "Tropical": (-1_500_000, 500_000),
    "S. Subtropical": (-3_500_000, -1_500_000),
    "S. Temperate": (-6_000_000, -3_500_000),
}

sample_cells = []
sample_labels = []
for name, (y_lo, y_hi) in BANDS.items():
    in_band = (coords[:, 1] >= y_lo) & (coords[:, 1] < y_hi)
    candidates = np.where(in_band)[0]
    if len(candidates) == 0:
        continue
    # Pick cell with most valid traits
    best = candidates[masks[candidates].sum(axis=1).argmax()]
    sample_cells.append(best)
    sample_labels.append(f"{name} (lat {lat[best]:.1f})")

print(f"Selected {len(sample_cells)} sample cells")
for label, idx in zip(sample_labels, sample_cells):
    print(f"  {label}: cell {idx}, valid traits = {masks[idx].sum()}")

In [ ]:
n_sample = len(sample_cells)
n_key = len(key_idx)

fig, axes = plt.subplots(
    n_sample, n_key, figsize=(3 * n_key, 2.5 * n_sample), squeeze=False,
)

for row, (cell_i, label) in enumerate(zip(sample_cells, sample_labels)):
    for col, (j, trait) in enumerate(key_idx):
        ax = axes[row, col]
        edges = bin_edges[j]
        centers = 0.5 * (edges[:-1] + edges[1:])
        widths = edges[1:] - edges[:-1]
        probs = histograms[cell_i, j, :]

        color = "#2ecc71" if masks[cell_i, j] else "#e74c3c"
        ax.bar(centers, probs, width=widths * 0.9, color=color, edgecolor="none")

        if row == 0:
            desc = trait_desc.get(trait, "")[:25]
            ax.set_title(f"{trait}\n{desc}", fontsize=7)
        if col == 0:
            ax.set_ylabel(label, fontsize=7)
        ax.tick_params(labelsize=5)
        ax.set_ylim(0, None)

fig.suptitle("Sample cell distributions (green=valid, red=invalid)", fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()